# Inference Router Validation

This notebook validates the smart load balancing strategy for LLM inference using:
- Real Mistral 7B inference via vLLM
- Alpaca instruction dataset (500 prompts)
- Ridge regression predictor for output token counts
- Rigorous A/B testing with statistical confidence intervals

**Expected runtime: 1.5-2.5 hours on RTX 4090**

## Setup

In [ ]:
import logging
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

from utils.datasets import load_alpaca
from utils.inference import VLLMInference
from utils.predictor import OutputLengthPredictor, PromptFeatureExtractor
from utils.load_balancer import SmartLoadBalancer, RoutingStrategy
from utils.statistics import calculate_statistics, summarize_results

logger.info("All imports successful")

## Configuration

In [ ]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Configuration:")
print(json.dumps(config, indent=2))

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
logger.info(f"Results will be saved to {results_dir}")

## Phase 1: Load Dataset

In [ ]:
num_prompts = config["dataset"]["num_prompts"]
prompts = load_alpaca(num_prompts)

print(f"Loaded {len(prompts)} prompts")
print(f"\nFirst 3 prompts:")
for i, p in enumerate(prompts[:3]):
    print(f"\n{i+1}. {p[:100]}...")

## Phase 2: Initialize vLLM

In [ ]:
model_config = config["model"]
vllm_config = config["vllm"]

logger.info(f"Initializing vLLM with {model_config['name']}")

inference = VLLMInference(
    model_name=model_config["name"],
    max_tokens=model_config["max_tokens"],
    batch_size=vllm_config["batch_size"],
    gpu_memory_utilization=vllm_config["gpu_memory_utilization"],
)

logger.info("vLLM initialized successfully")

## Phase 3: Collect Training Data & Train Predictor

In [ ]:
logger.info("Running inference to collect training data")
output_tokens = []

for i, prompt in enumerate(prompts):
    if (i + 1) % 50 == 0:
        logger.info(f"  Progress: {i+1}/{len(prompts)}")
    
    output, tokens = inference.generate(prompt)
    output_tokens.append(tokens)

logger.info(f"Inference complete. Mean output: {np.mean(output_tokens):.0f} tokens")
print(f"Output token stats: min={np.min(output_tokens)}, max={np.max(output_tokens)}, mean={np.mean(output_tokens):.0f}")

In [ ]:
logger.info("Training Ridge regression predictor")

predictor = OutputLengthPredictor(
    alpha=config["predictor"]["ridge_alpha"]
)

metrics = predictor.train(
    prompts=prompts,
    output_lengths=output_tokens,
    test_split=0.2,
    random_state=42
)

feature_importance = predictor.get_feature_importance()

print(f"\nPredictor Performance:")
print(f"  Test R²: {metrics['test_r2']:.3f}")
print(f"  Test MAE: {metrics['test_mae']:.1f} tokens")
print(f"\nFeature Importance:")
for fname, coef in feature_importance.items():
    print(f"  {fname}: {coef:.4f}")

## Phase 4: A/B Test Setup

In [ ]:
ab_config = config["ab_test"]
requests_per_strategy = ab_config["requests_per_strategy"]
threshold = config["smart_routing"]["cost_threshold"]

load_balancer = SmartLoadBalancer(num_workers=3)
extractor = PromptFeatureExtractor()

print(f"A/B Test Configuration:")
print(f"  Requests per strategy: {requests_per_strategy}")
print(f"  Routing threshold: {threshold} tokens")
print(f"  Number of workers: 3")

## Phase 5: Smart Routing A/B Test

In [ ]:
test_prompts = prompts[:requests_per_strategy * 2]

logger.info(f"Running A/B test with {requests_per_strategy} requests per strategy")
logger.info("Testing SMART_ROUTING strategy")

load_balancer.reset()
smart_latencies = []

for i in range(requests_per_strategy):
    prompt = test_prompts[i]
    
    features = extractor.extract_features(prompt)
    predicted_tokens = predictor.predict_single(features)
    
    worker_id = load_balancer.route_request(
        prompt_id=i,
        predicted_tokens=predicted_tokens,
        strategy=RoutingStrategy.PREDICTED_COST,
        threshold=threshold
    )
    
    output, tokens = inference.generate(prompt)
    latency = inference.last_latency
    smart_latencies.append(latency)
    
    load_balancer.record_completion(worker_id=worker_id, latency=latency)
    
    if (i + 1) % 50 == 0:
        logger.info(f"  Progress: {i+1}/{requests_per_strategy}")

logger.info(f"Smart routing complete. Mean latency: {np.mean(smart_latencies):.1f}ms")

## Phase 6: Round-Robin Baseline A/B Test

In [ ]:
logger.info("Testing ROUND_ROBIN strategy")

load_balancer.reset()
rr_latencies = []

for i in range(requests_per_strategy):
    prompt = test_prompts[requests_per_strategy + i]
    
    worker_id = load_balancer.route_request(
        prompt_id=i + requests_per_strategy,
        predicted_tokens=0,
        strategy=RoutingStrategy.ROUND_ROBIN,
        threshold=threshold
    )
    
    output, tokens = inference.generate(prompt)
    latency = inference.last_latency
    rr_latencies.append(latency)
    
    load_balancer.record_completion(worker_id=worker_id, latency=latency)
    
    if (i + 1) % 50 == 0:
        logger.info(f"  Progress: {i+1}/{requests_per_strategy}")

logger.info(f"Round-robin complete. Mean latency: {np.mean(rr_latencies):.1f}ms")

## Phase 7: Statistical Analysis

In [ ]:
logger.info("Calculating statistical metrics")

statistics = calculate_statistics(smart_latencies, rr_latencies)
statistics["predictor_r2"] = metrics["test_r2"]
statistics["predictor_mae"] = metrics["test_mae"]

summary = summarize_results(statistics)
print(summary)

results_file = results_dir / "validation_results.json"
with open(results_file, "w") as f:
    json.dump(statistics, f, indent=2)
logger.info(f"Results saved to {results_file}")

summary_file = results_dir / "validation_report.txt"
with open(summary_file, "w") as f:
    f.write(summary)
logger.info(f"Summary saved to {summary_file}")

## Phase 8: Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.hist(smart_latencies, bins=30, alpha=0.6, label="Smart Routing", color="blue")
ax.hist(rr_latencies, bins=30, alpha=0.6, label="Round-Robin", color="orange")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Frequency")
ax.set_title("Latency Distributions")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[0, 1]
percentiles = [50, 75, 90, 95, 99]
smart_percentiles = [np.percentile(smart_latencies, p) for p in percentiles]
rr_percentiles = [np.percentile(rr_latencies, p) for p in percentiles]

x = np.arange(len(percentiles))
width = 0.35
ax.bar(x - width/2, smart_percentiles, width, label="Smart Routing", color="blue")
ax.bar(x + width/2, rr_percentiles, width, label="Round-Robin", color="orange")
ax.set_xlabel("Percentile")
ax.set_ylabel("Latency (ms)")
ax.set_title("Percentile Comparison")
ax.set_xticks(x)
ax.set_xticklabels([f"P{p}" for p in percentiles])
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1, 0]
names = list(feature_importance.keys())
values = list(feature_importance.values())
ax.barh(names, values, color="green")
ax.set_xlabel("Coefficient")
ax.set_title("Predictor Feature Importance")
ax.grid(alpha=0.3, axis="x")

ax = axes[1, 1]
ax.axis("off")

comparison = statistics["comparison"]
metrics_text = f"""A/B Test Results

P95 Improvement: {comparison['p95_improvement_percent']:+.1f}%

p-value: {comparison['ttest']['p_value']:.4f}
Significant: {comparison['significant_at_0_05']}

Smart P95: {statistics['smart_routing']['p95']:.1f}ms
Round-Robin P95: {statistics['round_robin']['p95']:.1f}ms
"""

ax.text(0.1, 0.9, metrics_text, transform=ax.transAxes,
        fontsize=11, verticalalignment="top", family="monospace",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plot_file = results_dir / "validation_plots.png"
plt.savefig(plot_file, dpi=150, bbox_inches="tight")
logger.info(f"Plots saved to {plot_file}")
plt.show()

## Summary

In [ ]:
logger.info("Validation complete!")

if comparison["significant_at_0_05"]:
    print("\n✅ RESULT: Smart routing provides statistically significant improvement")
    print(f"   P95 latency improved by {comparison['p95_improvement_percent']:.1f}%")
else:
    print("\n⚠️  RESULT: No statistically significant difference found")
    print(f"   p-value = {comparison['ttest']['p_value']:.4f}")

print(f"\nResults saved to {results_dir}/")